# Testing Accuracy Notebook
This notebook completely skips training. It only loads the necessary classes, paths, and saved model weights (`checkpoint.pth`), then runs the final evaluation.

In [ ]:
import os
import pandas as pd
import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

## 1. Setup Audio Transformations
Define the fixed audio length (3 seconds) and the Log-Mel Spectrogram.

In [ ]:
TARGET_SR = 16000
TARGET_DURATION = 3
TARGET_LENGTH = TARGET_SR * TARGET_DURATION

def crop_or_pad(audio):
    length = len(audio)
    if length > TARGET_LENGTH:
        # Center crop for deterministic testing
        start = (length - TARGET_LENGTH) // 2
        audio = audio[start:start + TARGET_LENGTH]
    elif length < TARGET_LENGTH:
        pad_amount = TARGET_LENGTH - length
        audio = np.pad(audio, (0, pad_amount), mode='constant')
    return audio

mel_transform = T.MelSpectrogram(
    sample_rate=16000,
    n_fft=400,
    hop_length=160,
    n_mels=80
)

amplitude_to_db = T.AmplitudeToDB()

## 2. Define Dataset Class and Evaluation

In [ ]:
class SpeakerPairDataset(Dataset):
    def __init__(self, dataframe, mel_transform, amplitude_to_db):
        self.df = dataframe
        self.mel_transform = mel_transform
        self.amplitude_to_db = amplitude_to_db

    def __len__(self):
        return len(self.df)

    def load_audio(self, path):
        audio, sr = sf.read(path)
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        
        audio = crop_or_pad(audio)
        audio = torch.tensor(audio).float().unsqueeze(0)
        
        mel = self.mel_transform(audio)
        log_mel = self.amplitude_to_db(mel)
        return log_mel

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        mel1 = self.load_audio(row["path1_abs"])
        mel2 = self.load_audio(row["path2_abs"])
        label = torch.tensor(row["label"]).float()
        return mel1, mel2, label

def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    
    progress_bar = tqdm(loader, desc="Evaluating")
    
    with torch.no_grad():
        for mel1, mel2, labels in progress_bar:
            mel1 = mel1.to(device)
            mel2 = mel2.to(device)
            labels = labels.to(device)
            
            emb1 = model(mel1)
            emb2 = model(mel2)
            
            cosine_sim = F.cosine_similarity(emb1, emb2)
            preds = (cosine_sim > 0.5).float()
            
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
            progress_bar.set_postfix(acc=correct / total)
            
    print("\nFinal Accuracy:", correct / total)

## 3. Load Model weights from Kaggle Dataset
This loads `checkpoint.pth` from your 'Trained Configuration' dataset.

In [ ]:
class ResNetSpeaker(nn.Module):
    def __init__(self, embedding_dim=256):
        super().__init__()
        self.backbone = models.resnet18(weights=None)
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.embedding = nn.Linear(in_features, embedding_dim)

    def forward(self, x):
        features = self.backbone(x)
        embedding = self.embedding(features)
        embedding = F.normalize(embedding, p=2, dim=1)
        return embedding

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNetSpeaker(embedding_dim=256).to(device)

# Update this path to exactly where your checkpoint.pth is injected in kaggle/input/
# Based on your screenshot, it should be in "trained-configuration". 
CHECKPOINT_PATH = "/kaggle/input/datasets/tahmidulislamomi/trained-configuration/checkpoint.pth"

try:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint["model_state"])
    print("Model loaded perfectly from:", CHECKPOINT_PATH)
except Exception as e:
    print(f"Could not load checkpoint. Double check Kaggle path! Error: {e}")

## 4. Test Data Preparation
Load the separate test CSV and fix the paths to use the specific Kaggle dataset paths.

In [ ]:
TEST_CSV_PATH = "/kaggle/input/datasets/tahmidulislamomi/ml-project-testing-dataset/test_pairs.csv"
TEST_BASE_AUDIO_DIR = "/kaggle/input/datasets/tahmidulislamomi/ml-project-testing-dataset"

test_df = pd.read_csv(TEST_CSV_PATH)

def to_test_absolute_path(bad_path):
    cleaned = bad_path.replace("/kaggle/input/your-librispeech-dataset/", "", 1)
    return os.path.join(TEST_BASE_AUDIO_DIR, cleaned)

test_df["path1_abs"] = test_df["audio_path_1"].apply(to_test_absolute_path)
test_df["path2_abs"] = test_df["audio_path_2"].apply(to_test_absolute_path)

missing_1 = (~test_df["path1_abs"].apply(os.path.exists)).sum()
missing_2 = (~test_df["path2_abs"].apply(os.path.exists)).sum()
print(f"Missing path1: {missing_1} | Missing path2: {missing_2}")
if missing_1 > 0 or missing_2 > 0:
    print("WARNING: Some audio files are missing! Check TEST_BASE_AUDIO_DIR.")

## 5. Final Evaluation

In [ ]:
test_dataset = SpeakerPairDataset(test_df, mel_transform, amplitude_to_db)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

evaluate(model, test_loader, device)